# Preparing Data For Modeling — Wave 1


**Source dataset:** `9thCLEANED.csv` → **Output:** `Baseline_Dataset50.csv`

---

## Pipeline Overview

| Step | Description |
|---|---|
| 1 | Setup & load data |
| 2 | Encode nominal variables (One-Hot Encoding) |
| 3 | Inspect missing values |
| 4 | Imputation (mean, mode, KNN) |
| 5 | Drop identifier column |
| 6 | Export dataset |

In [1]:
import numpy as np
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

# ── Load raw cleaned dataset ──────────────────────────────────────────────────
wave1_model = pd.read_csv('9thCLEANED.csv')

print(f'  Shape : {wave1_model.shape}')
print(f'  Columns ({len(wave1_model.columns)}):')
print(wave1_model.columns.tolist())

  Shape : (15900, 39)
  Columns (39):
['X4EVERDROP', 'X1MTHEFF', 'X1SCIEFF', 'X1TXMTSCOR', 'S1M8GRADE', 'S1S8GRADE', 'X1PAREDU', 'X1PAREDEXPCT', 'X1SCHOOLBEL', 'S1PLAN', 'X1SCHOOLCLI', 'X1POVERTY', 'X1LOCALE', 'X1RACE', 'X1CONTROL', 'X1REGION', 'S1EDUEXPECT', 'S1SUREHSGRAD', 'X1FAMINCOME', 'X2SEX', 'S1FYAA', 'S1FYBA', 'S1FYLICENSE', 'S1FYAPPR', 'S1FYMILITARY', 'S1FYJOB', 'S1FYFAMILY', 'S1FYTRAVEL', 'S1FYVOLUN', 'S1FYNOTSURE', 'S1PAYOFF', 'S1GETINTOCLG', 'S1AFFORD', 'S1WORKING', 'STU_ID', 'X1PAR_SURVEY_MISSING', 'X1_DUAL_PARENT', 'MATH_NO_CLASS', 'SCI_NO_CLASS']


---
## 2. Encode Nominal Variables



Three columns have no inherent order and are encoded with **one-hot encoding**:

| Column | Description | Categories |
|---|---|---|
| `X1RACE` | Student race/ethnicity | 8 |
| `X1LOCALE` | School locale (City/Suburb/Town/Rural) | 4 |
| `X1REGION` | Geographic region | 4 |

`drop_first=False` is used to keep all categories — preferred for tree-based models
which do not suffer from the dummy variable trap.

In [2]:
# ── Quick inspection: unique values in nominal columns ────────────────────────
nominal_cols_to_encode = ['X1RACE', 'X1LOCALE', 'X1REGION']

for col in nominal_cols_to_encode:
    print(f"\n{'─' * 50}")
    print(f"  Column: {col}")
    print(f"{'─' * 50}")
    counts = wave1_model[col].value_counts(dropna=False).sort_index()
    total  = counts.sum()
    for val, count in counts.items():
        pct   = count / total * 100
        label = 'NaN' if pd.isna(val) else val
        print(f"  {label:>6} → {count:>6} ({pct:>5.1f}%)")
    print(f"  {'─' * 30}")
    print(f"  {'Total':>6} → {total:>6} (100.0%)")


──────────────────────────────────────────────────
  Column: X1RACE
──────────────────────────────────────────────────
     1.0 →    117 (  0.7%)
     2.0 →   1295 (  8.1%)
     3.0 →   1631 ( 10.3%)
     4.0 →    135 (  0.8%)
     5.0 →   2346 ( 14.8%)
     6.0 →   1394 (  8.8%)
     7.0 →     70 (  0.4%)
     8.0 →   8912 ( 56.1%)
  ──────────────────────────────
   Total →  15900 (100.0%)

──────────────────────────────────────────────────
  Column: X1LOCALE
──────────────────────────────────────────────────
     1.0 →   4584 ( 28.8%)
     2.0 →   5742 ( 36.1%)
     3.0 →   1823 ( 11.5%)
     4.0 →   3751 ( 23.6%)
  ──────────────────────────────
   Total →  15900 (100.0%)

──────────────────────────────────────────────────
  Column: X1REGION
──────────────────────────────────────────────────
     1.0 →   2491 ( 15.7%)
     2.0 →   4315 ( 27.1%)
     3.0 →   6370 ( 40.1%)
     4.0 →   2724 ( 17.1%)
  ──────────────────────────────
   Total →  15900 (100.0%)


In [3]:
# ── One-hot encode nominal columns ────────────────────────────────────────────
wave1_model[nominal_cols_to_encode] = wave1_model[nominal_cols_to_encode].astype('Int64')

wave1_model = pd.get_dummies(
    wave1_model,
    columns=nominal_cols_to_encode,
    drop_first=False,   # keep all categories (better for tree-based models)
    dtype=int           # store as 0/1 integers instead of booleans
)

# ── Confirm new columns ───────────────────────────────────────────────────────
new_cols = [col for col in wave1_model.columns
            if any(col.startswith(base + '_') for base in nominal_cols_to_encode)]

print(f"  New dummy columns added ({len(new_cols)}):")
for col in new_cols:
    print(f"    {col}")
print(f"\n  wave1_model shape after encoding: {wave1_model.shape}")

  New dummy columns added (16):
    X1RACE_1
    X1RACE_2
    X1RACE_3
    X1RACE_4
    X1RACE_5
    X1RACE_6
    X1RACE_7
    X1RACE_8
    X1LOCALE_1
    X1LOCALE_2
    X1LOCALE_3
    X1LOCALE_4
    X1REGION_1
    X1REGION_2
    X1REGION_3
    X1REGION_4

  wave1_model shape after encoding: (15900, 52)


---
## 3. Inspect Missing Values


Scan every column for NaN — only columns with missing values are printed.

In [4]:
# ── Full NaN scan ─────────────────────────────────────────────────────────────
for col in wave1_model.columns:
    if wave1_model[col].isna().any():
        print(f"--- {col} ---")
        print(wave1_model[col].value_counts(dropna=False).sort_index())
        print("\n" + "="*40 + "\n")

--- X1MTHEFF ---
X1MTHEFF
-2.92     216
-2.65      10
-2.53      15
-2.51      31
-2.48      23
         ... 
 1.21     144
 1.23      91
 1.34     480
 1.62    1872
 NaN     1476
Name: count, Length: 161, dtype: int64


--- X1SCIEFF ---
X1SCIEFF
-2.91     177
-2.60       8
-2.49      27
-2.48      12
-2.47      23
         ... 
 1.41      87
 1.42     139
 1.53     294
 1.83    1203
 NaN     2526
Name: count, Length: 149, dtype: int64


--- X1PAREDU ---
X1PAREDU
1.0     675
2.0    4266
3.0    1966
4.0    3302
5.0    1774
7.0     900
NaN    3017
Name: count, dtype: int64


--- X1PAREDEXPCT ---
X1PAREDEXPCT
1.0       32
2.0      834
3.0      104
4.0      882
5.0       95
6.0     3853
7.0       58
8.0     2737
9.0       29
10.0    2999
11.0    1260
NaN     3017
Name: count, dtype: int64


--- X1POVERTY ---
X1POVERTY
0.0    10977
1.0     1906
NaN     3017
Name: count, dtype: int64


--- X1FAMINCOME ---
X1FAMINCOME
1.0     1098
2.0     2255
3.0     2079
4.0     1933
5.0     1481
6.0     11

In [5]:
# ── Focused check: parent survey columns ─────────────────────────────────────
cols_to_check = ['X1PAREDU', 'X1PAREDEXPCT', 'X1POVERTY', 'X1FAMINCOME']

missing_check = pd.DataFrame({
    'Missing Count': wave1_model[cols_to_check].isnull().sum(),
    'Missing %'    : (wave1_model[cols_to_check].isnull().sum() / len(wave1_model) * 100).round(2),
    'Type'         : ['Ordinal', 'Ordinal', 'Binary', 'Ordinal']
})
print(missing_check)

              Missing Count  Missing %     Type
X1PAREDU               3017      18.97  Ordinal
X1PAREDEXPCT           3017      18.97  Ordinal
X1POVERTY              3017      18.97   Binary
X1FAMINCOME            3017      18.97  Ordinal


---
## 4. Imputation

Different variables require different strategies based on *why* values are missing.

| Variable | Strategy | Reason |
|---|---|---|
| `X1_DUAL_PARENT` | Fill NaN → `1` | Non-response — mode imputation |
| `X1MTHEFF`, `X1SCIEFF` | Fill NaN → column mean | Missing due to not taking the subject; continuous scores |
| `S1EDUEXPECT` | Recode `11` → `0` | "Don't know" sits outside the ordinal scale; recoded below it |
| `X1PAREDEXPCT` | Recode `11` → `0`; Fill NaN → `0` | "Don't know" and non-response both signal parental disengagement |
| `X1PAREDU`, `X1POVERTY`, `X1FAMINCOME` | KNN (k=5) | MNAR unit non-response; KNN uses correlated SES and academic features |

### 4a. Simple Imputations — `X1_DUAL_PARENT`, `X1MTHEFF`, `X1SCIEFF`

In [6]:
# ── X1_DUAL_PARENT: NaN → 1 (mode; assume dual-parent household) ─────────────
wave1_model['X1_DUAL_PARENT'] = wave1_model['X1_DUAL_PARENT'].fillna(1).astype(int)
print(f"  Imputed : X1_DUAL_PARENT → filled NaN with 1")

# ── X1MTHEFF, X1SCIEFF: NaN → column mean ────────────────────────────────────
for col in ['X1MTHEFF', 'X1SCIEFF']:
    mean_val = wave1_model[col].mean()
    wave1_model[col] = wave1_model[col].fillna(mean_val)
    print(f"  Imputed : {col} → filled NaN with mean ({mean_val:.4f})")

  Imputed : X1_DUAL_PARENT → filled NaN with 1
  Imputed : X1MTHEFF → filled NaN with mean (0.0753)
  Imputed : X1SCIEFF → filled NaN with mean (0.0660)


### 4b. Ordinal Expectation Variables — `S1EDUEXPECT` & `X1PAREDEXPCT`

Both variables share the same scale (1 = Less than HS → 10 = PhD, 11 = Don't know).

**The problem with value `11`:**
Keeping `11` as-is places "Don't know" *above* a PhD numerically — logically broken.
Recoding to `0` (bottom of scale) is also unjustified because bivariate analysis showed:
- `S1EDUEXPECT` "Don't know" → **17.72%** dropout (barely above 15.36% baseline)
- `X1PAREDEXPCT` "Don't know" → **21.35%** dropout (sits between Complete Assoc 22.22%
  and Start Bach 20.00% — clearly mid-scale, not bottom)

**Decision — recode `11` → `NaN` for both, then KNN impute:**
- Avoids imposing any rank that the data doesn't support
- KNN uses each student's academic and SES profile to estimate a personalized value
- For `X1PAREDEXPCT`, the 3,017 NaN (parent survey unit non-response) and 1,260
  "Don't know" are combined — both represent parents without a formed expectation,
  making them equivalent missing value types
- Consistent with the KNN strategy already applied to `X1PAREDU`, `X1POVERTY`,
  `X1FAMINCOME` — all imputed together in one block in Step 4c
- Preserves the strong ordinal correlation with dropout (−0.145 / −0.149) that
  would be lost entirely if these variables were treated as nominal

In [7]:
# ── Recode 11 (Don't Know) → NaN for both expectation variables ──────────────
# S1EDUEXPECT : no real NaN — only 11s need recoding
# X1PAREDEXPCT: 3,017 existing NaN + 1,260 "Don't Know" (11) → combined as NaN
wave1_model['S1EDUEXPECT']  = wave1_model['S1EDUEXPECT'].replace(11, np.nan)
wave1_model['X1PAREDEXPCT'] = wave1_model['X1PAREDEXPCT'].replace(11, np.nan)

# ── Confirm NaN counts before KNN ────────────────────────────────────────────
for col in ['S1EDUEXPECT', 'X1PAREDEXPCT']:
    n = wave1_model[col].isna().sum()
    pct = n / len(wave1_model) * 100
    print(f"  {col:20} → {n:,} NaN ({pct:.1f}%)")

  S1EDUEXPECT          → 3,274 NaN (20.6%)
  X1PAREDEXPCT         → 4,277 NaN (26.9%)


### 4b. Selecting KNN Context Features — Correlation Analysis



Before running KNN imputation, we identify which features are most appropriate
to use as context. A feature is included as a context variable only if:

- `|r| > 0.10` against **all five** imputation targets
- No missing values of its own (cannot be a context feature if it has NaN)
- Theoretically linked to family background or academic profile

The correlation is computed only on rows where the target is not NaN,
so the result reflects the true relationship in observed data.

In [8]:
# ── Correlation analysis — KNN context feature selection ──────────────────────
df_corr = wave1_model.copy()
df_corr['S1EDUEXPECT']  = df_corr['S1EDUEXPECT'].replace(11, np.nan)
df_corr['X1PAREDEXPCT'] = df_corr['X1PAREDEXPCT'].replace(11, np.nan)

cols_to_impute = ['X1PAREDU', 'X1POVERTY', 'X1FAMINCOME', 'S1EDUEXPECT', 'X1PAREDEXPCT']

# Exclude: imputation targets, target variable, ID, and MNAR flag
# X1PAR_SURVEY_MISSING excluded — perfectly collinear with missingness itself
exclude = cols_to_impute + ['X4EVERDROP', 'STU_ID', 'X1PAR_SURVEY_MISSING']
candidates = [
    c for c in df_corr.columns
    if df_corr[c].isna().sum() == 0
    and c not in exclude
]

# ── Compute absolute correlations ─────────────────────────────────────────────
results = {}
for target in cols_to_impute:
    subset     = df_corr[candidates + [target]].dropna(subset=[target])
    results[target] = subset[candidates].corrwith(subset[target]).abs().round(3)

corr_df = pd.DataFrame(results)

# ── Keep only features where ALL correlations >= 0.10 ────────────────────────
qualified = corr_df[corr_df.min(axis=1) >= 0.10].sort_values('X1PAREDU', ascending=False)

# ── Print clean table ─────────────────────────────────────────────────────────
print(f"  Context features where |r| >= 0.10 across ALL five targets:\n")
print(f"  {'Feature':<18} {'X1PAREDU':>10} {'X1POVERTY':>10} {'X1FAMINCOME':>12} {'S1EDUEXPECT':>12} {'X1PAREDEXPCT':>13}")
print(f"  {'─' * 70}")
for feat, row in qualified.iterrows():
    print(f"  {feat:<18} " + "  ".join(f"{row[t]:>10.3f}" for t in cols_to_impute))

print(f"\n  Total qualified context features: {len(qualified)}")

  Context features where |r| >= 0.10 across ALL five targets:

  Feature              X1PAREDU  X1POVERTY  X1FAMINCOME  S1EDUEXPECT  X1PAREDEXPCT
  ──────────────────────────────────────────────────────────────────────
  X1TXMTSCOR              0.374       0.211       0.321       0.346       0.369
  X1CONTROL               0.265       0.127       0.299       0.158       0.159
  S1AFFORD                0.262       0.183       0.275       0.238       0.205
  S1S8GRADE               0.253       0.168       0.231       0.312       0.308
  S1M8GRADE               0.233       0.151       0.207       0.289       0.300
  S1FYBA                  0.227       0.140       0.203       0.391       0.286
  S1PAYOFF                0.179       0.142       0.167       0.242       0.206
  S1GETINTOCLG            0.168       0.114       0.151       0.277       0.246
  S1WORKING               0.168       0.104       0.143       0.360       0.281
  S1SUREHSGRAD            0.164       0.143       0.165      

### 4c. KNN Imputation — All Five Variables



`X1PAREDU`, `X1POVERTY`, `X1FAMINCOME`, `S1EDUEXPECT`, and `X1PAREDEXPCT`
are imputed together in a single KNN block (k=5).

Context features were selected by the correlation analysis above —
only features with `|r| >= 0.10` across **all five** imputation targets
were kept (10 features total). Imputing all five variables together allows
KNN to leverage cross-variable relationships — for example, a student's
own expectation informs the parent's imputed value and vice versa.

**Context features used:**

| Feature | What it captures |
|---|---|
| `X1TXMTSCOR` | Math test score — strongest SES proxy |
| `X1CONTROL` | School type (public/private) — SES signal |
| `S1AFFORD` | Perceived ability to afford college |
| `S1S8GRADE` | 8th grade Science grade |
| `S1M8GRADE` | 8th grade Math grade |
| `S1FYBA` | Plans to get a Bachelor's degree |
| `S1PAYOFF` | Perceived school value |
| `S1GETINTOCLG` | Perceived college access |
| `S1WORKING` | Think working more important than studying|
| `S1SUREHSGRAD` | Certainty of graduating high school |

In [9]:
# ── KNN Imputation — all five variables ───────────────────────────────────────
knn_context = [
    'X1TXMTSCOR',    # r: 0.374 / 0.211 / 0.321 / 0.346 / 0.369
    'X1CONTROL',     # r: 0.265 / 0.127 / 0.299 / 0.158 / 0.159
    'S1AFFORD',      # r: 0.262 / 0.183 / 0.275 / 0.238 / 0.205
    'S1S8GRADE',     # r: 0.253 / 0.168 / 0.231 / 0.312 / 0.308
    'S1M8GRADE',     # r: 0.233 / 0.151 / 0.207 / 0.289 / 0.300
    'S1FYBA',        # r: 0.227 / 0.140 / 0.203 / 0.391 / 0.286
    'S1PAYOFF',      # r: 0.179 / 0.142 / 0.167 / 0.242 / 0.206
    'S1GETINTOCLG',  # r: 0.168 / 0.114 / 0.151 / 0.277 / 0.246
    'S1WORKING',     # r: 0.168 / 0.104 / 0.143 / 0.360 / 0.281
    'S1SUREHSGRAD',  # r: 0.164 / 0.143 / 0.165 / 0.348 / 0.238
]

cols_to_impute = [
    'X1PAREDU',
    'X1POVERTY',
    'X1FAMINCOME',
    'S1EDUEXPECT',
    'X1PAREDEXPCT',
]

# ── Build input matrix, fit, transform ───────────────────────────────────────
knn_input  = wave1_model[knn_context + cols_to_impute].copy()
imputer    = KNNImputer(n_neighbors=5)
knn_output = imputer.fit_transform(knn_input)

# ── Write imputed values back only ───────────────────────────────────────────
wave1_model[cols_to_impute] = knn_output[:, len(knn_context):]

# ── Round back to valid integers ─────────────────────────────────────────────
wave1_model['X1PAREDU']     = wave1_model['X1PAREDU'].round().astype(int)
wave1_model['X1FAMINCOME']  = wave1_model['X1FAMINCOME'].round().astype(int)
wave1_model['X1POVERTY']    = wave1_model['X1POVERTY'].round().clip(0, 1).astype(int)
wave1_model['S1EDUEXPECT']  = wave1_model['S1EDUEXPECT'].round().clip(1, 10).astype(int)
wave1_model['X1PAREDEXPCT'] = wave1_model['X1PAREDEXPCT'].round().clip(1, 10).astype(int)

print(f"  ✓ KNN imputed (k=5): {cols_to_impute}")
print(f"  ✓ Context features : {knn_context}")

# ── Confirm no remaining NaN ──────────────────────────────────────────────────
remaining = wave1_model[cols_to_impute].isnull().sum()
if remaining.sum() == 0:
    print("\n  ✓ No remaining NaN in imputed columns")
else:
    print(f"\n  ⚠ Remaining NaN:\n{remaining}")

  ✓ KNN imputed (k=5): ['X1PAREDU', 'X1POVERTY', 'X1FAMINCOME', 'S1EDUEXPECT', 'X1PAREDEXPCT']
  ✓ Context features : ['X1TXMTSCOR', 'X1CONTROL', 'S1AFFORD', 'S1S8GRADE', 'S1M8GRADE', 'S1FYBA', 'S1PAYOFF', 'S1GETINTOCLG', 'S1WORKING', 'S1SUREHSGRAD']

  ✓ No remaining NaN in imputed columns


In [10]:
# ── Inspect unique values for parent survey columns ───────────────────────────
cols_to_impute = ['X1PAREDU', 'X1PAREDEXPCT', 'X1POVERTY', 'X1FAMINCOME']

for col in cols_to_impute:
    print(f"\n{'='*50}")
    print(f"  {col}")
    print(f"{'='*50}")
    print(wave1_model[col].value_counts(dropna=False).sort_index())


  X1PAREDU
X1PAREDU
1     684
2    5094
3    3365
4    3927
5    1904
6      25
7     901
Name: count, dtype: int64

  X1PAREDEXPCT
X1PAREDEXPCT
1       32
2      858
3      243
4     1249
5      713
6     4920
7      944
8     3546
9      316
10    3079
Name: count, dtype: int64

  X1POVERTY
X1POVERTY
0    13774
1     2126
Name: count, dtype: int64

  X1FAMINCOME
X1FAMINCOME
1     1111
2     2656
3     2891
4     2692
5     1967
6     1470
7      943
8      666
9      330
10     197
11     261
12     103
13     613
Name: count, dtype: int64


### 4d. Post-Imputation Fix — `X1PAREDU` Category 6


`X1PAREDU` has no category `6` in its original scale (jumps from 5 → 7).
KNN interpolation produced values that rounded to 6 — remapped to `5`
(Master's degree), the nearest valid lower category.

In [11]:
# ── Fix X1PAREDU: remap 6 → 5 (scale is 1,2,3,4,5,7 — no 6 exists) ──────────
wave1_model['X1PAREDU'] = wave1_model['X1PAREDU'].replace(6, 5)

print("  X1PAREDU value counts after fix:")
print(wave1_model['X1PAREDU'].value_counts().sort_index())

  X1PAREDU value counts after fix:
X1PAREDU
1     684
2    5094
3    3365
4    3927
5    1929
7     901
Name: count, dtype: int64


---
## 5. Drop Identifier Column — `STU_ID` & Redundant Columns



`STU_ID` is a unique student identifier used for record-keeping only.
It carries no predictive information and must be removed before modelling
to prevent the model from learning spurious patterns from an arbitrary ID number.

`S1PLAN` — redundant with the retained `S1FY*` post-high-school plan
  indicators and ordinal expectation features which capture the same
  information more directly and with stronger dropout signal

In [12]:
# ── Drop STU_ID ───────────────────────────────────────────────────────────────
wave1_model.drop(columns=['STU_ID'], inplace=True)

print(f"  Dropped  : STU_ID")
print(f"  Shape now  : {wave1_model.shape}")

  Dropped  : STU_ID
  Shape now  : (15900, 51)


In [13]:
# ── Drop S1PLAN ───────────────────────────────────────────────────────────────
wave1_model.drop(columns=['S1PLAN'], inplace=True)
print(f"  Dropped  : S1PLAN")
print(f"  Shape now  : {wave1_model.shape}")

  Dropped  : S1PLAN
  Shape now  : (15900, 50)


---
## 6. Export Dataset



Final NaN check before saving, then export `wave1_model` as
`Baseline_Dataset50.csv` — this is the fully encoded and imputed
dataset ready for feature selection and modelling.

In [14]:
# ── Final sanity check ────────────────────────────────────────────────────────
print(f"  Shape          : {wave1_model.shape}")
print(f"  NaN remaining  : {wave1_model.isnull().sum().sum()}")
print(f"  Duplicate rows : {wave1_model.duplicated().sum()}")
print(f"  Target balance : {wave1_model['X4EVERDROP'].value_counts().to_dict()}")

# ── Export ────────────────────────────────────────────────────────────────────
wave1_model.to_csv("Baseline_Dataset50.csv", index=False)
print(f"\n  ✓ Saved → Baseline_Dataset50.csv")

  Shape          : (15900, 50)
  NaN remaining  : 0
  Duplicate rows : 0
  Target balance : {0.0: 13457, 1.0: 2443}

  ✓ Saved → Baseline_Dataset50.csv


In [15]:
wave1_model.shape

(15900, 50)

# Feature Selection For Modeling


## Feature Selection

Feature selection was guided by three criteria established during EDA (Steps 3–6):
Spearman correlation with the target (`X4EVERDROP`), bivariate dropout rate gaps across
categories, and multicollinearity between predictors. Since multiple model types will be
compared (Logistic Regression, Decision Tree, XGBoost), a conservative approach was taken
— redundant features were resolved in favour of the least collinear and most interpretable variable.

---

### Dropped Features

| Feature | Reason |
|---|---|
| `X1FAMINCOME` | Highest multicollinearity node in the SES trio — r=0.597 with `X1PAREDU` and r=−0.583 with `X1POVERTY`. Dropping it breaks both correlated pairs at once while retaining the more interpretable SES indicators |
| `X1LOCALE` | Weak signal — small dropout rate gap across City/Suburb/Town/Rural categories, not informative enough to justify 4 dummy columns |
| `X1REGION` | Weak signal — geographic region shows no meaningful dropout gradient in bivariate analysis |
| `X2SEX` | Weak signal — minimal dropout rate difference between groups |
| `S1FYAA` | At or below the 15.4% baseline dropout rate — no meaningful signal |
| `S1FYBA` | At or below the 15.4% baseline dropout rate — no meaningful signal |
| `S1FYLICENSE` | At or below the 15.4% baseline dropout rate — no meaningful signal |
| `S1FYAPPR` | At or below the 15.4% baseline dropout rate — no meaningful signal |
| `S1FYJOB` | Marginal signal at 18.49% — likely redundant with SES features
| `S1FYTRAVEL` | At or below the 15.4% baseline dropout rate — no meaningful signal |
| `S1FYVOLUN` | At or below the 15.4% baseline dropout rate — no meaningful signal |
| `S1FYNOTSURE` | Redundant with `S1SUREHSGRAD` which captures graduation certainty more precisely |
| `STU_ID` | Student identifier — carries no predictive information |
| `S1PLAN` | Redundant — the retained `S1FY*` indicators and ordinal expectation features capture post-HS plans more directly |

---

### Retained Features & Justification

**Continuous** — all 5 retained; normally distributed standardised scores with consistent
dropout gaps between groups in bivariate analysis (Step 3 & 4):

| Feature | Justification |
|---|---|
| `X1TXMTSCOR` | Strongest continuous predictor (r=−0.260) — clear distributional separation between dropout and non-dropout groups |
| `X1MTHEFF` | Math self-efficacy — moderate signal; students who drop out rate themselves lower |
| `X1SCIEFF` | Science self-efficacy — moderate signal; mirrors math efficacy pattern |
| `X1SCHOOLBEL` | School belonging — dropout students consistently score lower; captures disengagement |
| `X1SCHOOLCLI` | School climate — negative mean (−0.36) across the sample; small but consistent dropout gap |

**Ordinal** — all retained based on strong monotonic dropout gradients confirmed in Steps 4–5:

| Feature | Justification |
|---|---|
| `S1M8GRADE` | r=+0.229 with target — strong monotonic gradient from 8% (A) to 40%+ (D/below); one of the top 3 predictors |
| `S1S8GRADE` | r=+0.242 — strongest ordinal correlate; mirrors math grade pattern across science |
| `X1PAREDU` | r=−0.205 — students with lower parent education drop out at nearly double the rate of those with college-educated parents |
| `X1PAREDEXPCT` | Strong dropout gradient across expectation levels; "Don't know" category shows elevated risk |
| `S1EDUEXPECT` | Among the strongest predictors — students expecting only HS or less show 3–5× above-baseline dropout rates |
| `S1SUREHSGRAD` | r=+0.228 — "Not at all sure" group likely exceeds 50% dropout; drives the grades × certainty interaction |
| `S1PAYOFF` | Pessimism about school value correlates with higher dropout; part of the college attitude cluster |
| `S1GETINTOCLG` | Low perceived college access is a consistent dropout risk signal |
| `S1AFFORD` | Perceived inability to afford college correlates with dropout, especially under poverty |
| `S1WORKING` | Student thinks working is more important than college|

**Binary** — retained based on dropout rate gaps and structural importance:

| Feature | Justification |
|---|---|
| `X1PAR_SURVEY_MISSING` | MNAR flag — parents who skipped the survey are associated with 24.3% dropout vs. 13.3% for respondents (+11pp); must be retained as a feature |
| `X1_DUAL_PARENT` | Household structure — single-parent households show higher dropout risk |
| `X1POVERTY` | Universal risk amplifier — poverty raises dropout rates across every racial group, education level, and expectation tier confirmed in multivariate analysis |
| `MATH_NO_CLASS` | Structural missingness flag — students with no 9th grade Math course drop out at 22.2% (+7pp above baseline) |
| `SCI_NO_CLASS` | Structural missingness flag — students with no 9th grade Science course drop out at 20.5% (+6pp above baseline) |
| `S1FYFAMILY` | Strongest post-HS plan signal at 25.16% dropout — family obligations in 9th grade directly compete with schooling |
| `S1FYMILITARY` | 23.24% dropout rate — distinct high-risk path not captured by any other retained feature |



In [16]:
# ============================================================
# Final Feature Set & Export
# ============================================================

continuous  = [
    'X1TXMTSCOR',   # Math assessment score — strongest continuous predictor
    'X1MTHEFF',     # Math self-efficacy
    'X1SCIEFF',     # Science self-efficacy
    'X1SCHOOLBEL',  # School belonging
    'X1SCHOOLCLI',  # School climate
]

ordinal     = [
    'S1M8GRADE',    # 8th grade Math grade — strong monotonic dropout gradient
    'S1S8GRADE',    # 8th grade Science grade — strongest ordinal correlate
    'X1PAREDU',     # Parent education level
    'X1PAREDEXPCT', # Parent educational expectation for their student
    'S1EDUEXPECT',  # Student educational expectation for themselves
    'S1SUREHSGRAD', # Certainty of graduating high school
    'S1PAYOFF',     # Perceived school value
    'S1GETINTOCLG', # Perceived college access
    'S1AFFORD',     # Perceived ability to afford college
    'S1WORKING',    # Student thinks working is more important than college
]

binary      = [
    'X1PAR_SURVEY_MISSING', # MNAR flag — non-response linked to 24.3% dropout
    'X1_DUAL_PARENT',       # Household structure - 1 or 2 parents/gaurdians
    'X1POVERTY',            # Below poverty line — universal risk amplifier
    'MATH_NO_CLASS',        # No 9th grade Math — 22.2% dropout rate
    'SCI_NO_CLASS',         # No 9th grade Science — 20.5% dropout rate
    'S1FYFAMILY',           # Plans to help family — 25.16% dropout rate
    'S1FYMILITARY',         # Plans military — 23.24% dropout rate
    'X1CONTROL',            # School type — Public 17.8% vs Private 4.8% dropout
]

target      = ['X4EVERDROP']

all_features = continuous + ordinal + binary + target

# ── Verify all columns exist ──────────────────────────────────
missing_cols = [c for c in all_features if c not in wave1_model.columns]
if missing_cols:
    print(f"  ⚠ Columns not found in dataset: {missing_cols}")
else:
    print(f"  ✓ All {len(all_features)} columns verified")

# ── Build final dataframe ─────────────────────────────────────
wave1_final = wave1_model[all_features].copy()

# ── Summary ───────────────────────────────────────────────────
print(f"\n  {'─' * 50}")
print(f"  Continuous       : {len(continuous)} features")
print(f"  Ordinal          : {len(ordinal)} features")
print(f"  Binary           : {len(binary)} features")
print(f"  Target           : X4EVERDROP")
print(f"  {'─' * 50}")
print(f"  Total features   : {len(all_features) - 1} (excluding target)")
print(f"  Final shape      : {wave1_final.shape}")

# ── Export ────────────────────────────────────────────────────
wave1_final.to_csv('Wave1_FeatureSelected.csv', index=False)
print(f"\n  ✓ Saved → Wave1_FeatureSelected.csv")

  ✓ All 24 columns verified

  ──────────────────────────────────────────────────
  Continuous       : 5 features
  Ordinal          : 10 features
  Binary           : 8 features
  Target           : X4EVERDROP
  ──────────────────────────────────────────────────
  Total features   : 23 (excluding target)
  Final shape      : (15900, 24)

  ✓ Saved → Wave1_FeatureSelected.csv


In [17]:
wave1_final.columns

Index(['X1TXMTSCOR', 'X1MTHEFF', 'X1SCIEFF', 'X1SCHOOLBEL', 'X1SCHOOLCLI',
       'S1M8GRADE', 'S1S8GRADE', 'X1PAREDU', 'X1PAREDEXPCT', 'S1EDUEXPECT',
       'S1SUREHSGRAD', 'S1PAYOFF', 'S1GETINTOCLG', 'S1AFFORD', 'S1WORKING',
       'X1PAR_SURVEY_MISSING', 'X1_DUAL_PARENT', 'X1POVERTY', 'MATH_NO_CLASS',
       'SCI_NO_CLASS', 'S1FYFAMILY', 'S1FYMILITARY', 'X1CONTROL',
       'X4EVERDROP'],
      dtype='object')

In [18]:
wave1_final.shape

(15900, 24)